# 00 Init

In [ ]:
import os
from dataclasses import dataclass
from typing import Dict, List, Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# 01 Mock Data

In [ ]:
# 单商家 mock 数据
merchant_df = pd.DataFrame([
    {
        "merchant_id": "M001",
        "merchant_name": "Blue Bottle Demo Store",
        "category": "coffee",
        "district": "Pudong",
        "orders_7d": 120,
        "orders_prev_7d": 145,
        "orders_last_30d_pctl": 0.94,  # 历史异常分位（MVP 先 mock）
        "orders_peer_pctl": 0.78,      # 同行异常分位（MVP 先 mock）
        "impressions": 5200,
        "conversion_rate": 0.021,
        "image_coverage": 0.46,
        "open_hours": 8.5,
        "accept_time_mins": 3.2,
        "prep_time_mins": 18.0,
        "merchant_cancel_rate": 0.075,
        "active_spu_count": 22,
        "discount_rate": 0.06,
    },
    {
        "merchant_id": "M002",
        "merchant_name": "Sunrise Burger Demo Store",
        "category": "burger",
        "district": "Xuhui",
        "orders_7d": 280,
        "orders_prev_7d": 270,
        "orders_last_30d_pctl": 0.55,
        "orders_peer_pctl": 0.42,
        "impressions": 10200,
        "conversion_rate": 0.034,
        "image_coverage": 0.81,
        "open_hours": 12.0,
        "accept_time_mins": 2.1,
        "prep_time_mins": 12.5,
        "merchant_cancel_rate": 0.021,
        "active_spu_count": 41,
        "discount_rate": 0.11,
    }
])

# 同类商家 benchmark 分布（MVP 用随机数模拟）
np.random.seed(42)

peer_benchmark = {
    "conversion_rate": np.clip(np.random.normal(0.03, 0.005, 200), 0.005, 0.2),
    "image_coverage": np.clip(np.random.normal(0.78, 0.12, 200), 0.05, 1.0),
    "open_hours": np.clip(np.random.normal(11.0, 1.8, 200), 4.0, 20.0),
    "accept_time_mins": np.clip(np.random.normal(2.5, 0.6, 200), 0.5, 10.0),
    "prep_time_mins": np.clip(np.random.normal(14.0, 2.5, 200), 5.0, 40.0),
    "merchant_cancel_rate": np.clip(np.random.normal(0.03, 0.012, 200), 0.0, 0.25),
    "active_spu_count": np.clip(np.random.normal(35, 8, 200), 1, 200),
    "discount_rate": np.clip(np.random.normal(0.10, 0.03, 200), 0.0, 0.6),
}

merchant_df.head()

# 02 Inputs

In [ ]:
merchant_id = "M001"
question = "这家商家最近经营怎么样？我应该建议他做什么？"

merchant_row = merchant_df.loc[merchant_df["merchant_id"] == merchant_id].iloc[0]
merchant_row

# 03 Configs

In [ ]:
HEALTH_CONFIG = {
    "warning_threshold": 0.70,
    "risk_hist_threshold": 0.90,
    "risk_peer_threshold": 0.70,
}

GAP_CONFIG = {
    "gap_percentile_threshold": 0.30,
    "strong_gap_percentile_threshold": 0.15,
}

# 展示顺序固定
DISPLAY_ORDER = [
    "traffic",
    "ops_readiness",
    "user_experience",
    "supply_quality",
]

# 决策权重动态
DRIVER_WEIGHTS = {
    "traffic": 0.35,
    "ops_readiness": 0.25,
    "user_experience": 0.20,
    "supply_quality": 0.20,
}

METRIC_TO_DRIVER = {
    "conversion_rate": "traffic",
    "image_coverage": "ops_readiness",
    "open_hours": "ops_readiness",
    "accept_time_mins": "ops_readiness",
    "prep_time_mins": "user_experience",
    "merchant_cancel_rate": "user_experience",
    "active_spu_count": "supply_quality",
    "discount_rate": "supply_quality",
}

ACTION_MAP = {
    "conversion_rate": "优化商品图片或设置更有吸引力的折扣",
    "image_coverage": "补齐菜单图片，优先补热销商品图",
    "open_hours": "延长晚间营业时间，覆盖高峰时段",
    "accept_time_mins": "优化接单流程，缩短接单时长",
    "prep_time_mins": "优化后厨流程，缩短出餐时间",
    "merchant_cancel_rate": "减少商责取消，优化备货与接单判断",
    "active_spu_count": "补充动销 SKU，优化菜单供给结构",
    "discount_rate": "提升核心商品折扣力度，增强价格竞争力",
}

# True = 越大越好；False = 越小越好
METRIC_DIRECTION = {
    "conversion_rate": True,
    "image_coverage": True,
    "open_hours": True,
    "accept_time_mins": False,
    "prep_time_mins": False,
    "merchant_cancel_rate": False,
    "active_spu_count": True,
    "discount_rate": True,
}

# 04 Data sturcture

In [ ]:
@dataclass
class HealthResult:
    status: str
    hist_percentile: float
    peer_percentile: float
    summary: str


@dataclass
class GapResult:
    metric: str
    driver: str
    percentile: float
    value: float
    label: str


@dataclass
class ActionResult:
    driver: str
    metric: str
    action: str
    score: float
    priority: str

# 05 Helper Functions

In [ ]:
def percentile_rank(value: float, population: np.ndarray, higher_is_better: bool = True) -> float:
    """
    返回 value 在 population 中的相对分位（0~1）。
    对于 higher_is_better=False 的指标，会自动反转，使得“表现越差 → percentile 越低”。
    """
    population = np.asarray(population)
    raw = float((population < value).mean())
    return raw if higher_is_better else 1 - raw


def gap_severity(percentile: float) -> float:
    """
    把 percentile 映射成 gap 严重程度。
    percentile 越低，severity 越高。
    """
    return max(0.0, 1.0 - percentile)


def priority_label(score: float) -> str:
    if score >= 0.60:
        return "High"
    if score >= 0.35:
        return "Medium"
    return "Low"

# 06 Health Model

In [ ]:
def calculate_growth_health(row: pd.Series, config: Dict[str, float]) -> HealthResult:
    hist_p = float(row["orders_last_30d_pctl"])
    peer_p = float(row["orders_peer_pctl"])

    if hist_p >= config["risk_hist_threshold"] and peer_p >= config["risk_peer_threshold"]:
        status = "Risk"
    elif hist_p >= config["warning_threshold"] or peer_p >= config["warning_threshold"]:
        status = "Warning"
    else:
        status = "Healthy"

    growth = (row["orders_7d"] - row["orders_prev_7d"]) / row["orders_prev_7d"]
    summary = (
        f"近7日订单变化 {growth:.1%}；"
        f"历史异常分位 {hist_p:.0%}；"
        f"同行异常分位 {peer_p:.0%}。"
    )

    return HealthResult(
        status=status,
        hist_percentile=hist_p,
        peer_percentile=peer_p,
        summary=summary,
    )


health_result = calculate_growth_health(merchant_row, HEALTH_CONFIG)
health_result

# 07 Gap model

In [ ]:
def identify_gaps(
    row: pd.Series,
    peer_data: Dict[str, np.ndarray],
    gap_config: Dict[str, float],
) -> List[GapResult]:
    gap_results: List[GapResult] = []

    for metric, driver in METRIC_TO_DRIVER.items():
        value = float(row[metric])
        higher_is_better = METRIC_DIRECTION[metric]
        p = percentile_rank(value, peer_data[metric], higher_is_better=higher_is_better)

        if p < gap_config["strong_gap_percentile_threshold"]:
            label = "Strong Gap"
        elif p < gap_config["gap_percentile_threshold"]:
            label = "Gap"
        elif p > 0.70:
            label = "Strong"
        else:
            label = "Normal"

        gap_results.append(
            GapResult(
                metric=metric,
                driver=driver,
                percentile=p,
                value=value,
                label=label,
            )
        )

    return gap_results


gap_results = identify_gaps(merchant_row, peer_benchmark, GAP_CONFIG)
pd.DataFrame([g.__dict__ for g in gap_results]).sort_values(["driver", "percentile"])

# 08 Action Engine

In [ ]:
def generate_actions(gaps: List[GapResult]) -> List[ActionResult]:
    actions: List[ActionResult] = []

    for gap in gaps:
        if gap.label not in {"Gap", "Strong Gap"}:
            continue

        severity = gap_severity(gap.percentile)
        score = severity * DRIVER_WEIGHTS[gap.driver]

        actions.append(
            ActionResult(
                driver=gap.driver,
                metric=gap.metric,
                action=ACTION_MAP[gap.metric],
                score=score,
                priority=priority_label(score),
            )
        )

    actions = sorted(actions, key=lambda x: x.score, reverse=True)
    return actions


action_results = generate_actions(gap_results)
pd.DataFrame([a.__dict__ for a in action_results])

# 09 Sturctured output

In [ ]:
def build_structured_output(
    merchant: pd.Series,
    health: HealthResult,
    gaps: List[GapResult],
    actions: List[ActionResult],
    question: str,
) -> Dict[str, Any]:
    grouped_gaps: Dict[str, List[Dict[str, Any]]] = {driver: [] for driver in DISPLAY_ORDER}

    for g in gaps:
        grouped_gaps[g.driver].append({
            "metric": g.metric,
            "value": g.value,
            "percentile": round(g.percentile, 3),
            "label": g.label,
        })

    return {
        "merchant_id": merchant["merchant_id"],
        "merchant_name": merchant["merchant_name"],
        "question": question,
        "health": health.__dict__,
        "gaps_by_driver": grouped_gaps,
        "recommended_actions": [a.__dict__ for a in actions],
    }


structured_output = build_structured_output(
    merchant=merchant_row,
    health=health_result,
    gaps=gap_results,
    actions=action_results,
    question=question,
)

structured_output

# 10 Human Readable Output

In [ ]:
def render_report(output: Dict[str, Any]) -> None:
    print("=== Merchant Growth Copilot ===")
    print(f"Merchant: {output['merchant_name']} ({output['merchant_id']})")
    print(f"Question: {output['question']}")
    print()

    print("[1] Health Status")
    print(f"- Status: {output['health']['status']}")
    print(f"- Summary: {output['health']['summary']}")
    print()

    print("[2] Key Gaps by Driver")
    for driver in DISPLAY_ORDER:
        print(f"- {driver}")
        items = output["gaps_by_driver"][driver]
        if not items:
            print("  - No metrics")
            continue

        for item in items:
            print(
                f"  - {item['metric']}: {item['label']} "
                f"(value={item['value']}, percentile={item['percentile']:.1%})"
            )
    print()

    print("[3] Recommended Actions")
    if not output["recommended_actions"]:
        print("- No recommended actions")
    else:
        for action in output["recommended_actions"]:
            print(
                f"- [{action['priority']}] {action['action']} "
                f"(driver={action['driver']}, metric={action['metric']}, score={action['score']:.2f})"
            )


render_report(structured_output)

# 11 LLM Explanation Layer 

In [ ]:
def build_llm_prompt(output: Dict[str, Any]) -> str:
    return f"""
你是一个帮助客户经理做商家经营分析的 AI 助手。

请根据以下结构化分析结果，输出：
1. 一段 executive summary
2. 2-3 条关键问题
3. 2-3 条建议动作

结构化结果：
{output}
""".strip()


prompt_preview = build_llm_prompt(structured_output)
print(prompt_preview[:1500])

# 12 OpenAI key 

In [ ]:
# 先不要急着运行这段；等你把前面逻辑跑通再接 API

# from openai import OpenAI
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# response = client.responses.create(
#     model="gpt-4.1",
#     input=build_llm_prompt(structured_output),
# )

# print(response.output_text)